In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Converting json and xml in csv.

In [ ]:
import json
import csv

INPUT_JSON = "/content/drive/MyDrive/BMKG_project/nvdcve-2.0-2025.json"
OUTPUT_CSV = "/content/drive/MyDrive/BMKG_project/cve_flat.csv"


def get_english_description(descriptions):
    for d in descriptions or []:
        if d.get("lang") == "en":
            return d.get("value")
    return None

def normalize_name(name):

    if not name:
        return None

    name = name.replace("_", " ")
    name = " ".join(w.capitalize() for w in name.split())
    return name

def extract_cvss(metrics):

    if not metrics:
        return None, None, None, None, None

    if metrics.get("cvssMetricV31"):
        m = metrics["cvssMetricV31"][0]
        data = m.get("cvssData", {})

        return (
            data.get("version", "3.1"),
            data.get("baseScore"),
            data.get("baseSeverity"),
            m.get("exploitabilityScore"),
            m.get("impactScore"),
        )

    if metrics.get("cvssMetricV2"):
        m = metrics["cvssMetricV2"][0]
        data = m.get("cvssData", {})

        return (
            data.get("version", "2.0"),
            data.get("baseScore"),
            m.get("baseSeverity"),
            m.get("exploitabilityScore"),
            m.get("impactScore"),
        )

    return None, None, None, None, None


def extract_weaknesses(weaknesses):

    cwes = []

    for w in weaknesses or []:
        for d in w.get("description", []):
            val = d.get("value")
            if isinstance(val, str) and val.startswith("CWE-"):
                cwes.append(val.strip())

    return sorted(set(cwes))


def parse_cpe(criteria):

    if not isinstance(criteria, str):
        return None, None, None

    parts = criteria.split(":")

    if len(parts) < 6:
        return None, None, None

    vendor = parts[3]
    product = parts[4]
    version = parts[5]

    if version in ["*", "-"]:
        version = None

    return vendor, product, version


def extract_configurations(configurations):

    vendors = set()
    products = set()
    versions = set()

    version_start = set()
    version_end = set()

    pairs = set()

    for conf in configurations or []:

        for node in conf.get("nodes", []):

            for match in node.get("cpeMatch", []):

                # consider only vulnerable matches
                if match.get("vulnerable") is not True:
                    continue

                criteria = match.get("criteria")

                vendor, product, version = parse_cpe(criteria)

                vendor = normalize_name(vendor)
                product = normalize_name(product)

                if vendor:
                    vendors.add(vendor)

                if product:
                    products.add(product)

                if vendor and product:
                    pairs.add((vendor, product))

                if version:
                    versions.add(version)

                if match.get("versionStartIncluding"):
                    version_start.add(match["versionStartIncluding"])

                if match.get("versionStartExcluding"):
                    version_start.add(match["versionStartExcluding"])

                if match.get("versionEndIncluding"):
                    version_end.add(match["versionEndIncluding"])

                if match.get("versionEndExcluding"):
                    version_end.add(match["versionEndExcluding"])

    return {
        "vendors": sorted(vendors),
        "products": sorted(products),
        "versions": sorted(versions),
        "version_start": sorted(version_start),
        "version_end": sorted(version_end),
        "pairs": sorted(list(pairs)),
    }


fieldnames = [
    "cve_id",
    "source_identifier",
    "vuln_status",
    "published",
    "last_modified",
    "description_en",
    "cvss_version",
    "base_score",
    "base_severity",
    "exploitability_score",
    "impact_score",
    "weaknesses",
    "vendors",
    "products",
    "product_vendor_pairs",
    "versions",
    "version_start",
    "version_end",
]


with open(INPUT_JSON, "r", encoding="utf-8") as f:

    data = json.load(f)

    vulns = data["vulnerabilities"]

    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8-sig") as out:

        writer = csv.DictWriter(out, fieldnames=fieldnames)
        writer.writeheader()

        for item in vulns:

            cve = item.get("cve", {})

            description_en = get_english_description(cve.get("descriptions"))

            cvss_version, base_score, base_severity, exploit_score, impact_score = extract_cvss(
                cve.get("metrics")
            )

            weaknesses = extract_weaknesses(cve.get("weaknesses"))

            configs = extract_configurations(cve.get("configurations"))

            row = {

                "cve_id": cve.get("id"),
                "source_identifier": cve.get("sourceIdentifier"),
                "vuln_status": cve.get("vulnStatus"),
                "published": cve.get("published"),
                "last_modified": cve.get("lastModified"),
                "description_en": description_en,

                "cvss_version": cvss_version,
                "base_score": base_score,
                "base_severity": base_severity,
                "exploitability_score": exploit_score,
                "impact_score": impact_score,

                "weaknesses": json.dumps(weaknesses),

                "vendors": json.dumps(configs["vendors"]),
                "products": json.dumps(configs["products"]),
                "product_vendor_pairs": json.dumps(configs["pairs"]),

                "versions": json.dumps(configs["versions"]),
                "version_start": json.dumps(configs["version_start"]),
                "version_end": json.dumps(configs["version_end"]),
            }

            writer.writerow(row)

print("CSV creato:", OUTPUT_CSV)

CSV creato: /content/drive/MyDrive/BMKG_project/cve_flat.csv


In [ ]:
import pandas as pd

cve_path = "/content/drive/MyDrive/BMKG_project/cve_flat.csv"

df = pd.read_csv(cve_path)

print("Totale CVE nel dataset:", len(df))

Totale CVE nel dataset: 43200


In [ ]:
import xml.etree.ElementTree as ET
import csv

xml_file = "/content/drive/MyDrive/BMKG_project/cwec_v4.19.1.xml"

tree = ET.parse(xml_file)
root = tree.getroot()

ns = {"cwe": "http://cwe.mitre.org/cwe-7"}

with open("/content/drive/MyDrive/BMKG_project/cwe_full.csv", "w", newline="", encoding="utf-8") as f:

    writer = csv.writer(f)

    writer.writerow([
        "cwe_id",
        "name",
        "description",
        "extended_description",
        "likelihood",
        "parents",
        "platforms",
        "detection_methods",
        "mitigations",
        "consequences"
    ])

    for weakness in root.findall(".//cwe:Weakness", ns):

        cwe_id = f"CWE-{weakness.get('ID')}"
        name = weakness.get("Name")

        desc_elem = weakness.find("cwe:Description", ns)
        description = desc_elem.text.strip() if desc_elem is not None and desc_elem.text else ""

        ext_elem = weakness.find("cwe:Extended_Description", ns)
        extended_description = ext_elem.text.strip() if ext_elem is not None and ext_elem.text else ""

        likelihood_elem = weakness.find("cwe:Likelihood_Of_Exploit", ns)
        likelihood = likelihood_elem.text.strip() if likelihood_elem is not None and likelihood_elem.text else ""

        # --- Parents ---
        parents = []
        for rel in weakness.findall(".//cwe:Related_Weakness", ns):
            if rel.get("Nature") == "ChildOf":
                parents.append(f"CWE-{rel.get('CWE_ID')}")

        parents_str = "|".join(parents)

        # --- Platforms ---
        platforms = []
        for lang in weakness.findall(".//cwe:Language", ns):
            value = lang.get("Name") or lang.get("Class")
            if value:
                platforms.append(f"Language:{value}")

        for tech in weakness.findall(".//cwe:Technology", ns):
            value = tech.get("Name") or tech.get("Class")
            if value:
                platforms.append(f"Technology:{value}")

        platforms_str = "|".join(platforms)

        # --- Detection Methods ---
        detection_methods = []
        for dm in weakness.findall(".//cwe:Detection_Method", ns):
            method_elem = dm.find("cwe:Method", ns)
            eff_elem = dm.find("cwe:Effectiveness", ns)

            method = method_elem.text.strip() if method_elem is not None and method_elem.text else ""
            effectiveness = eff_elem.text.strip() if eff_elem is not None and eff_elem.text else ""

            if method:
                detection_methods.append(f"{method} ({effectiveness})")

        detection_str = "|".join(detection_methods)

        # --- Mitigations ---
        mitigations = []
        for mit in weakness.findall(".//cwe:Mitigation", ns):
            desc_elem = mit.find("cwe:Description", ns)
            eff_elem = mit.find("cwe:Effectiveness", ns)

            mitigation_text = desc_elem.text.strip() if desc_elem is not None and desc_elem.text else ""
            effectiveness = eff_elem.text.strip() if eff_elem is not None and eff_elem.text else ""

            if mitigation_text:
                mitigations.append(f"{mitigation_text} ({effectiveness})")

        mitigations_str = "|".join(mitigations)

        # --- Consequences ---
        consequences = []
        for cons in weakness.findall(".//cwe:Consequence", ns):
            scope_elem = cons.find("cwe:Scope", ns)
            impact_elem = cons.find("cwe:Impact", ns)

            scope = scope_elem.text.strip() if scope_elem is not None and scope_elem.text else ""
            impact = impact_elem.text.strip() if impact_elem is not None and impact_elem.text else ""

            if scope or impact:
                consequences.append(f"{scope}:{impact}")

        consequences_str = "|".join(consequences)

        writer.writerow([
            cwe_id,
            name,
            description,
            extended_description,
            likelihood,
            parents_str,
            platforms_str,
            detection_str,
            mitigations_str,
            consequences_str
        ])

print("Single CWE CSV created: cwe_full.csv")

Single CWE CSV created: cwe_full.csv


## Building graph

In [ ]:
!pip -q install rdflib
!pip -q install rdflib pyshacl pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.8/109.8 kB 2.6 MB/s eta 0:00:00


In [25]:
import pandas as pd
import json
import re
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, OWL, XSD, DCTERMS

CVE_PATH = "/content/drive/MyDrive/BMKG_Project_i6457170/data/cve_flat.csv"
CWE_PATH = "/content/drive/MyDrive/BMKG_Project_i6457170/data/cwe_full.csv"
KEV_PATH = "/content/drive/MyDrive/BMKG_Project_i6457170/data/known_exploited_vulnerabilities.csv"

df_cve = pd.read_csv(CVE_PATH, encoding="utf-8-sig")
df_cwe = pd.read_csv(CWE_PATH, sep=",", encoding="utf-8-sig")
df_kev = pd.read_csv(KEV_PATH, sep=";", encoding="latin1")


print("CVE:", df_cve.shape)
print("CWE:", df_cwe.shape)
print("KEV:", df_kev.shape)

CVE: (43200, 18)
CWE: (969, 10)
KEV: (1527, 11)


In [ ]:
BASE = "http://example.org/vulnkg/"

VULN = Namespace(BASE + "vocab/")
CVE  = Namespace(BASE + "cve/")
CWE  = Namespace(BASE + "cwe/")
PROD = Namespace(BASE + "product/")
ORG  = Namespace(BASE + "org/")
KEV  = Namespace(BASE + "kev/")
PROC = Namespace(BASE + "process/")

SCHEMA = Namespace("https://schema.org/")
SKOS   = Namespace("http://www.w3.org/2004/02/skos/core#")
PROV = Namespace("http://www.w3.org/ns/prov#")

g = Graph()

g.bind("vuln", VULN)
g.bind("cve", CVE)
g.bind("cwe", CWE)
g.bind("prod", PROD)
g.bind("org", ORG)
g.bind("kev", KEV)
g.bind("schema", SCHEMA)
g.bind("skos", SKOS)
g.bind("dcterms", DCTERMS)
g.bind("owl", OWL)
g.bind("rdfs", RDFS)
g.bind("xsd", XSD)
g.bind("prov", PROV)
g.bind("proc", PROC)

In [ ]:
# HELPERS

def slug(text):
    if pd.isna(text) or text is None:
        return None
    text = str(text).strip().lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text or None

def parse_json_list(value):
    if pd.isna(value) or value is None:
        return []
    s = str(value).strip()
    if s == "" or s == "[]" or s.lower() == "nan":
        return []
    try:
        parsed = json.loads(s)
        return parsed if isinstance(parsed, list) else []
    except Exception:
        return []

def parse_datetime(value):
    if pd.isna(value) or value is None or str(value).strip() == "":
        return None
    try:
        dt = pd.to_datetime(value, dayfirst=True, errors="coerce")
        if pd.isna(dt):
            return None
        return dt.isoformat()
    except Exception:
        return None

def parse_date(value):
    if pd.isna(value) or value is None or str(value).strip() == "":
        return None
    try:
        dt = pd.to_datetime(value, dayfirst=True, errors="coerce")
        if pd.isna(dt):
            return None
        return dt.date().isoformat()
    except Exception:
        return None

def add_literal_if_present(graph, s, p, value, datatype=None, lang=None):
    if pd.isna(value) or value is None or str(value).strip() == "":
        return
    if lang is not None:
        graph.add((s, p, Literal(str(value), lang=lang)))
    elif datatype is not None:
        graph.add((s, p, Literal(value, datatype=datatype)))
    else:
        graph.add((s, p, Literal(str(value))))

def add_decimal_if_present(graph, s, p, value):
    if pd.isna(value) or value is None or str(value).strip() == "":
        return
    try:
        dec = Decimal(str(value))
        graph.add((s, p, Literal(dec, datatype=XSD.decimal)))
    except (InvalidOperation, ValueError):
        pass

In [ ]:
# Classes
for cls in [
    VULN.Vulnerability,
    VULN.Weakness,
    VULN.KEVRecord,
    SCHEMA.SoftwareApplication,
    SCHEMA.Organization
]:
    g.add((cls, RDF.type, OWL.Class))

# Object properties
for prop, dom, ran in [
    (VULN.hasWeakness, VULN.Vulnerability, VULN.Weakness),
    (VULN.affectsProduct, VULN.Vulnerability, SCHEMA.SoftwareApplication),
    (VULN.aboutVulnerability, VULN.KEVRecord, VULN.Vulnerability),
    (VULN.kevHasWeakness, VULN.KEVRecord, VULN.Weakness),
    (SCHEMA.manufacturer, SCHEMA.SoftwareApplication, SCHEMA.Organization),
]:
    g.add((prop, RDF.type, OWL.ObjectProperty))
    g.add((prop, RDFS.domain, dom))
    g.add((prop, RDFS.range, ran))



# Vulnerability datatype properties
for prop in [
    VULN.vulnStatus,
    VULN.baseScore,
    VULN.baseSeverity,
    VULN.exploitabilityScore,
    VULN.impactScore,
]:
    g.add((prop, RDF.type, OWL.DatatypeProperty))
    g.add((prop, RDFS.domain, VULN.Vulnerability))

# Weakness datatype properties
for prop in [
    DCTERMS.description,
    VULN.likelihoodOfExploit,
]:
    g.add((prop, RDF.type, OWL.DatatypeProperty))
    g.add((prop, RDFS.domain, VULN.Weakness))

# KEV datatype properties
for prop in [
    VULN.kevDateAdded,
    VULN.kevVulnerabilityName,
]:
    g.add((prop, RDF.type, OWL.DatatypeProperty))
    g.add((prop, RDFS.domain, VULN.KEVRecord))

# PROVENANCE RESOURCES

NVD_AGENT = ORG.NVD
CWE_AGENT = ORG.MITRE_CWE
KEV_AGENT = ORG.CISA_KEV
KG_ACTIVITY = PROC.GraphIntegrationProcess

# Agents
for agent_uri, label in [
    (NVD_AGENT, "National Vulnerability Database"),
    (CWE_AGENT, "MITRE Common Weakness Enumeration"),
    (KEV_AGENT, "CISA Known Exploited Vulnerabilities Catalog"),
]:
    g.add((agent_uri, RDF.type, PROV.Agent))
    g.add((agent_uri, RDFS.label, Literal(label, lang="en")))

# Activity
g.add((KG_ACTIVITY, RDF.type, PROV.Activity))
g.add((KG_ACTIVITY, RDFS.label, Literal("Knowledge graph construction process", lang="en")))
g.add((KG_ACTIVITY, PROV.used, NVD_AGENT))
g.add((KG_ACTIVITY, PROV.used, CWE_AGENT))
g.add((KG_ACTIVITY, PROV.used, KEV_AGENT))

<Graph identifier=N0ae67c5079a344dea401e7528a833fd2 (<class 'rdflib.graph.Graph'>)>

In [ ]:
# BUILD CWE + HIERARCHY

# First pass: valid non-deprecated CWE IDs
valid_cwe_ids = set()

for _, row in df_cwe.iterrows():
    cwe_id = row.get("cwe_id")
    name = row.get("name")

    if pd.isna(cwe_id):
        continue

    cwe_id = str(cwe_id).strip()
    if cwe_id == "":
        continue

    name_str = "" if pd.isna(name) else str(name).strip()
    if name_str.upper().startswith("DEPRECATED"):
        continue

    valid_cwe_ids.add(cwe_id)

# Second pass: add only valid CWE nodes
for _, row in df_cwe.iterrows():
    cwe_id = row.get("cwe_id")
    if pd.isna(cwe_id):
        continue

    cwe_id = str(cwe_id).strip()
    if cwe_id == "" or cwe_id not in valid_cwe_ids:
        continue

    cwe_uri = URIRef(CWE[cwe_id])

    g.add((cwe_uri, RDF.type, VULN.Weakness))
    g.add((cwe_uri, DCTERMS.identifier, Literal(cwe_id)))

    # Provenance
    g.add((cwe_uri, PROV.wasAttributedTo, CWE_AGENT))
    g.add((cwe_uri, PROV.wasGeneratedBy, KG_ACTIVITY))

    name = row.get("name")
    if pd.isna(name) or str(name).strip() == "":
        name = cwe_id
    g.add((cwe_uri, SKOS.prefLabel, Literal(str(name), lang="en")))

    add_literal_if_present(g, cwe_uri, DCTERMS.description, row.get("description"), lang="en")
    add_literal_if_present(g, cwe_uri, VULN.likelihoodOfExploit, row.get("likelihood"), datatype=XSD.string)

    parents = row.get("parents")
    if not pd.isna(parents):
        for p in re.split(r"[|,;]\s*", str(parents)):
            p = p.strip()
            if p.startswith("CWE-") and p in valid_cwe_ids:
                g.add((cwe_uri, SKOS.broader, URIRef(CWE[p])))

print("Triples after CWE block:", len(g))
print("Valid non-deprecated CWE count:", len(valid_cwe_ids))

Triples after CWE block: 7057
Valid non-deprecated CWE count: 944


In [ ]:
# BUILD CVE + PRODUCT + ORG
from decimal import Decimal, InvalidOperation
seen_products = set()
seen_orgs = set()

for _, row in df_cve.iterrows():
    cve_id = row.get("cve_id")
    if pd.isna(cve_id):
        continue

    cve_id = str(cve_id).strip()
    if cve_id == "":
        continue

    cve_uri = URIRef(CVE[cve_id])

    g.add((cve_uri, RDF.type, VULN.Vulnerability))
    g.add((cve_uri, DCTERMS.identifier, Literal(cve_id)))
    g.add((cve_uri, SKOS.prefLabel, Literal(cve_id)))

    # Provenance
    g.add((cve_uri, PROV.wasAttributedTo, NVD_AGENT))
    g.add((cve_uri, PROV.wasGeneratedBy, KG_ACTIVITY))

    add_literal_if_present(g, cve_uri, DCTERMS.description, row.get("description_en"), lang="en")

    issued = parse_datetime(row.get("published"))
    if issued:
        g.add((cve_uri, DCTERMS.issued, Literal(issued, datatype=XSD.dateTime)))

    add_literal_if_present(g, cve_uri, VULN.vulnStatus, row.get("vuln_status"), datatype=XSD.string)

    add_decimal_if_present(g, cve_uri, VULN.baseScore, row.get("base_score"))
    add_literal_if_present(g, cve_uri, VULN.baseSeverity, row.get("base_severity"), datatype=XSD.string)
    add_decimal_if_present(g, cve_uri, VULN.exploitabilityScore, row.get("exploitability_score"))
    add_decimal_if_present(g, cve_uri, VULN.impactScore, row.get("impact_score"))

    # NVD weaknesses
    weaknesses = parse_json_list(row.get("weaknesses"))
    for w in weaknesses:
        if isinstance(w, str):
            w = w.strip()
            if w.startswith("CWE-") and w in valid_cwe_ids:
                g.add((cve_uri, VULN.hasWeakness, URIRef(CWE[w])))

    # Product/vendor links
    pairs = parse_json_list(row.get("product_vendor_pairs"))
    for pair in pairs:
        if not isinstance(pair, list) or len(pair) != 2:
            continue

        vendor_name, product_name = pair
        vendor_slug = slug(vendor_name)
        product_slug = slug(product_name)

        if not vendor_slug or not product_slug:
            continue

        org_uri = URIRef(ORG[vendor_slug])
        prod_uri = URIRef(PROD[f"{vendor_slug}/{product_slug}"])

        if org_uri not in seen_orgs:
            g.add((org_uri, RDF.type, SCHEMA.Organization))
            g.add((org_uri, SCHEMA.name, Literal(str(vendor_name))))
            seen_orgs.add(org_uri)

        if prod_uri not in seen_products:
            g.add((prod_uri, RDF.type, SCHEMA.SoftwareApplication))
            g.add((prod_uri, SCHEMA.name, Literal(str(product_name))))
            g.add((prod_uri, SCHEMA.manufacturer, org_uri))
            seen_products.add(prod_uri)

        g.add((cve_uri, VULN.affectsProduct, prod_uri))

print("Triples after CVE block:", len(g))

/tmp/ipykernel_7019/3520821986.py:27: UserWarning: Parsing dates in %Y-%m-%dT%H:%M:%S.%f format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  dt = pd.to_datetime(value, dayfirst=True, errors="coerce")


Triples after CVE block: 645982


In [ ]:
# BUILD KEV RECORDS
valid_cve_ids = set(
    str(x).strip()
    for x in df_cve["cve_id"]
    if pd.notna(x) and str(x).strip() != ""
)

unmatched_kev_ids = set()

for _, row in df_kev.iterrows():
    cve_id = row.get("cveID")
    if pd.isna(cve_id):
        continue

    cve_id = str(cve_id).strip()
    if cve_id == "":
        continue

    if cve_id not in valid_cve_ids:
        unmatched_kev_ids.add(cve_id)
        continue

    cve_uri = URIRef(CVE[cve_id])
    kev_uri = URIRef(KEV[cve_id])

    g.add((kev_uri, RDF.type, VULN.KEVRecord))
    g.add((kev_uri, VULN.aboutVulnerability, cve_uri))
    g.add((kev_uri, DCTERMS.identifier, Literal(cve_id)))
    g.add((kev_uri, SKOS.prefLabel, Literal(f"KEV record for {cve_id}")))

    # Provenance
    g.add((kev_uri, PROV.wasAttributedTo, KEV_AGENT))
    g.add((kev_uri, PROV.wasGeneratedBy, KG_ACTIVITY))

    date_added = parse_date(row.get("dateAdded"))
    if date_added:
        g.add((kev_uri, VULN.kevDateAdded, Literal(date_added, datatype=XSD.date)))

    add_literal_if_present(g, kev_uri, DCTERMS.description, row.get("shortDescription"), datatype=XSD.string)
    add_literal_if_present(g, kev_uri, VULN.kevVulnerabilityName, row.get("vulnerabilityName"), datatype=XSD.string)

    cwes_raw = row.get("cwes")
    if pd.isna(cwes_raw) or str(cwes_raw).strip() == "":
        cwes = []
    else:
        cwes = [x.strip() for x in str(cwes_raw).split(",") if x.strip()]

    for cwe in cwes:
        if cwe.startswith("CWE-") and cwe in valid_cwe_ids:
            g.add((kev_uri, VULN.kevHasWeakness, URIRef(CWE[cwe])))

In [ ]:
OUT_TTL = "/content/drive/MyDrive/BMKG_project/vulnerability_fixed_kg.ttl"
g.serialize(destination=OUT_TTL, format="turtle")
print("Saved:", OUT_TTL)
print("Total triples:", len(g))

Saved: /content/drive/MyDrive/BMKG_project/vulnerability_fixed_kg.ttl
Total triples: 647627


## SHACL constraint

In [ ]:
!pip -q install pyshacl

In [ ]:
shapes_ttl = """
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
@prefix vuln: <http://example.org/vulnkg/vocab/> .
@prefix schema: <https://schema.org/> .
@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix skos: <http://www.w3.org/2004/02/skos/core#> .

# Vulnerability

vuln:VulnerabilityShape
    a sh:NodeShape ;
    sh:targetClass vuln:Vulnerability ;

    sh:property [
        sh:path dcterms:identifier ;
        sh:datatype xsd:string ;
        sh:minCount 1 ;
        sh:maxCount 1 ;
    ] ;

    sh:property [
        sh:path dcterms:description ;
        sh:nodeKind sh:Literal ;
        sh:maxCount 1 ;
    ] ;

    sh:property [
        sh:path dcterms:issued ;
        sh:datatype xsd:dateTime ;
        sh:maxCount 1 ;
    ] ;

    sh:property [
        sh:path vuln:vulnStatus ;
        sh:datatype xsd:string ;
        sh:maxCount 1 ;
    ] ;

    sh:property [
        sh:path vuln:baseScore ;
        sh:datatype xsd:decimal ;
        sh:minInclusive "0.0"^^xsd:decimal ;
        sh:maxInclusive "10.0"^^xsd:decimal ;
        sh:maxCount 1 ;
    ] ;

    sh:property [
        sh:path vuln:baseSeverity ;
        sh:datatype xsd:string ;
        sh:maxCount 1 ;
    ] ;

    sh:property [
        sh:path vuln:exploitabilityScore ;
        sh:datatype xsd:decimal ;
        sh:minInclusive "0.0"^^xsd:decimal ;
        sh:maxInclusive "10.0"^^xsd:decimal ;
        sh:maxCount 1 ;
    ] ;

    sh:property [
        sh:path vuln:impactScore ;
        sh:datatype xsd:decimal ;
        sh:minInclusive "0.0"^^xsd:decimal ;
        sh:maxInclusive "10.0"^^xsd:decimal ;
        sh:maxCount 1 ;
    ] ;

    sh:property [
        sh:path vuln:hasWeakness ;
        sh:class vuln:Weakness ;
    ] ;

    sh:property [
        sh:path vuln:affectsProduct ;
        sh:class schema:SoftwareApplication ;
    ] .

# Weakness

vuln:WeaknessShape
    a sh:NodeShape ;
    sh:targetClass vuln:Weakness ;

    sh:property [
        sh:path dcterms:identifier ;
        sh:datatype xsd:string ;
        sh:minCount 1 ;
        sh:maxCount 1 ;
    ] ;

    sh:property [
        sh:path skos:prefLabel ;
        sh:nodeKind sh:Literal ;
        sh:minCount 1 ;
        sh:maxCount 1 ;
    ] ;

    sh:property [
        sh:path dcterms:description ;
        sh:nodeKind sh:Literal ;
        sh:maxCount 1 ;
    ] ;

    sh:property [
        sh:path vuln:likelihoodOfExploit ;
        sh:datatype xsd:string ;
        sh:maxCount 1 ;
    ] ;

    sh:property [
        sh:path skos:broader ;
        sh:class vuln:Weakness ;
    ] .

# SoftwareApplication

vuln:SoftwareApplicationShape
    a sh:NodeShape ;
    sh:targetClass schema:SoftwareApplication ;

    sh:property [
        sh:path schema:name ;
        sh:nodeKind sh:Literal ;
        sh:minCount 1 ;
        sh:maxCount 1 ;
    ] ;

    sh:property [
        sh:path schema:manufacturer ;
        sh:class schema:Organization ;
        sh:minCount 1 ;
        sh:maxCount 1 ;
    ] .

# Organization

vuln:OrganizationShape
    a sh:NodeShape ;
    sh:targetClass schema:Organization ;

    sh:property [
        sh:path schema:name ;
        sh:nodeKind sh:Literal ;
        sh:minCount 1 ;
        sh:maxCount 1 ;
    ] .

# KEVRecord

vuln:KEVRecordShape
    a sh:NodeShape ;
    sh:targetClass vuln:KEVRecord ;

    sh:property [
        sh:path vuln:aboutVulnerability ;
        sh:class vuln:Vulnerability ;
        sh:minCount 1 ;
        sh:maxCount 1 ;
    ] ;

    sh:property [
        sh:path vuln:kevDateAdded ;
        sh:datatype xsd:date ;
        sh:maxCount 1 ;
    ] ;

    sh:property [
        sh:path dcterms:description ;
        sh:nodeKind sh:Literal ;
        sh:maxCount 1 ;
    ] ;

    sh:property [
        sh:path vuln:kevVulnerabilityName ;
        sh:datatype xsd:string ;
        sh:maxCount 1 ;
    ] ;

    sh:property [
        sh:path vuln:kevHasWeakness ;
        sh:class vuln:Weakness ;
    ] .
"""

In [ ]:
from rdflib import Graph
from pyshacl import validate

shapes_graph = Graph()
shapes_graph.parse(data=shapes_ttl, format="turtle")

conforms, results_graph, results_text = validate(
    data_graph=g,
    shacl_graph=shapes_graph,
    inference="rdfs",
    abort_on_first=False,
    allow_infos=True,
    allow_warnings=True
)

print("Conforms:", conforms)
print(results_text)

Conforms: True
Validation Report
Conforms: True



## Query SPARQL

In [ ]:
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, OWL, XSD, DCTERMS
g=Graph()
g.parse("/content/drive/MyDrive/BMKG_Project_i6457170/data/vulnerability_kg.ttl")

<Graph identifier=Nf854de918c02462e8b6a40b692c988f8 (<class 'rdflib.graph.Graph'>)>

In [ ]:
import pandas as pd

def run_sparql_to_df(graph: Graph, query: str) -> pd.DataFrame:
    res = graph.query(query)
    cols = [str(v) for v in res.vars]
    rows = []
    for r in res:
        rows.append([str(x) if x is not None else "" for x in r])
    return pd.DataFrame(rows, columns=cols)

In [ ]:
ent="""
SELECT (COUNT(DISTINCT ?entity) AS ?nEntities)
WHERE {
    ?entity a ?type .
}

"""
run_sparql_to_df(g, ent)

,nEntities
0,56098


In [ ]:
vuln="""
PREFIX vuln: <http://example.org/vulnkg/vocab/>

SELECT (COUNT(DISTINCT ?cve) AS ?nVulnerabilities)
WHERE {
    ?cve a vuln:Vulnerability .
}
"""
run_sparql_to_df(g, vuln)

,nVulnerabilities
0,43200


In [ ]:
weak="""
PREFIX vuln: <http://example.org/vulnkg/vocab/>

SELECT (COUNT(DISTINCT ?cwe) AS ?nWeaknesses)
WHERE {
    ?cwe a vuln:Weakness .
}
"""
run_sparql_to_df(g, weak)

,nWeaknesses
0,944


In [ ]:
kev="""
PREFIX vuln: <http://example.org/vulnkg/vocab/>

SELECT (COUNT(DISTINCT ?kev) AS ?nKEVRecords)
WHERE {
    ?kev a vuln:KEVRecord .
}
"""
run_sparql_to_df(g, kev)

,nKEVRecords
0,165


In [ ]:
softapp="""
PREFIX schema: <https://schema.org/>

SELECT (COUNT(DISTINCT ?product) AS ?nProducts)
WHERE {
    ?product a schema:SoftwareApplication .
}
"""
run_sparql_to_df(g, softapp)

,nProducts
0,8670


In [ ]:
org="""
PREFIX schema: <https://schema.org/>

SELECT (COUNT(DISTINCT ?org) AS ?nOrganizations)
WHERE {
    ?org a schema:Organization .
}
"""
run_sparql_to_df(g, org)

,nOrganizations
0,3096


In [ ]:
trip="""
SELECT (COUNT(*) AS ?nTriples)
WHERE {
    ?s ?p ?o .
}
"""
run_sparql_to_df(g, trip)

,nTriples
0,647627


In [ ]:
#different weakness categories between KEV and NVD
diff="""
PREFIX vuln: <http://example.org/vulnkg/vocab/>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX dcterms: <http://purl.org/dc/terms/>

SELECT ?cveId ?nvdCWE ?nvdLabel ?kevCWE ?kevLabel
WHERE {

    ?kev a vuln:KEVRecord ;
         vuln:aboutVulnerability ?cve ;
         vuln:kevHasWeakness ?kevCWE .

    ?cve dcterms:identifier ?cveId .

    OPTIONAL {
        ?cve vuln:hasWeakness ?nvdCWE .
        ?nvdCWE skos:prefLabel ?nvdLabel .
    }

    ?kevCWE skos:prefLabel ?kevLabel .

    FILTER NOT EXISTS {
        ?cve vuln:hasWeakness ?kevCWE .
    }
}
ORDER BY ?cveId
"""
run_sparql_to_df(g, diff)

,cveId,nvdCWE,nvdLabel,kevCWE,kevLabel
0,CVE-2025-27920,http://example.org/vulnkg/cwe/CWE-24,Path Traversal: '../filedir',http://example.org/vulnkg/cwe/CWE-22,Improper Limitation of a Pathname to a Restric...
1,CVE-2025-32706,http://example.org/vulnkg/cwe/CWE-20,Improper Input Validation,http://example.org/vulnkg/cwe/CWE-122,Heap-based Buffer Overflow
2,CVE-2025-32756,http://example.org/vulnkg/cwe/CWE-121,Stack-based Buffer Overflow,http://example.org/vulnkg/cwe/CWE-124,Buffer Underwrite ('Buffer Underflow')
3,CVE-2025-32756,http://example.org/vulnkg/cwe/CWE-787,Out-of-bounds Write,http://example.org/vulnkg/cwe/CWE-124,Buffer Underwrite ('Buffer Underflow')
4,CVE-2025-3935,http://example.org/vulnkg/cwe/CWE-502,Deserialization of Untrusted Data,http://example.org/vulnkg/cwe/CWE-287,Improper Authentication
5,CVE-2025-47827,http://example.org/vulnkg/cwe/CWE-347,Improper Verification of Cryptographic Signature,http://example.org/vulnkg/cwe/CWE-324,Use of a Key Past its Expiration Date


In [ ]:
#weaknesses families and the highest average score
q_family_score = """
PREFIX vuln: <http://example.org/vulnkg/vocab/>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?parentCWE ?parentLabel
       (COUNT(DISTINCT ?cve) AS ?nVulns)
       (AVG(xsd:decimal(?score)) AS ?avgBaseScore)
WHERE {
    ?cve a vuln:Vulnerability ;
         vuln:hasWeakness ?childCWE ;
         vuln:baseScore ?score .

    ?childCWE skos:broader ?parentCWE .
    ?parentCWE skos:prefLabel ?parentLabel .
}
GROUP BY ?parentCWE ?parentLabel
HAVING (COUNT(DISTINCT ?cve) >= 10)
ORDER BY DESC(?avgBaseScore)
LIMIT 3
"""
run_sparql_to_df(g, q_family_score)

,parentCWE,parentLabel,nVulns,avgBaseScore
0,http://example.org/vulnkg/cwe/CWE-602,Client-Side Enforcement of Server-Side Security,11,8.3
1,http://example.org/vulnkg/cwe/CWE-190,Integer Overflow or Wraparound,11,8.272727272727272727272727273
2,http://example.org/vulnkg/cwe/CWE-306,Missing Authentication for Critical Function,156,8.226282051282051282051282051


In [ ]:
#product and vendor most affected by exploited vulnerabilities
prodmostaff= """
PREFIX vuln: <http://example.org/vulnkg/vocab/>
PREFIX schema: <https://schema.org/>

SELECT ?vendorName ?productName (COUNT(DISTINCT ?cve) AS ?nExploitedVulns)
WHERE {
    ?kev a vuln:KEVRecord ;
         vuln:aboutVulnerability ?cve .

    ?cve vuln:affectsProduct ?product .
    ?product schema:name ?productName ;
             schema:manufacturer ?vendor .

    ?vendor schema:name ?vendorName .
}
GROUP BY ?vendorName ?productName
ORDER BY DESC(?nExploitedVulns)
LIMIT 1
"""
run_sparql_to_df(g, prodmostaff)

,vendorName,productName,nExploitedVulns
0,Microsoft,Windows Server 2022 23h2,25


In [ ]:
#Most common weaknesses in exploited vulnerabilities
weak_in_exp="""
PREFIX vuln: <http://example.org/vulnkg/vocab/>
PREFIX cve: <http://example.org/vulnkg/cve/>
PREFIX cwe: <http://example.org/vulnkg/cwe/>
PREFIX schema: <https://schema.org/>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX dcterms: <http://purl.org/dc/terms/>

SELECT ?cwe ?name (COUNT(?cve) AS ?n)
WHERE {

    ?kev a vuln:KEVRecord ;
         vuln:aboutVulnerability ?cve .

    ?cve vuln:hasWeakness ?cwe .

    ?cwe skos:prefLabel ?name .

}
GROUP BY ?cwe ?name
ORDER BY DESC(?n)
LIMIT 10

"""
run_sparql_to_df(g, weak_in_exp)

,cwe,name,n
0,http://example.org/vulnkg/cwe/CWE-502,Deserialization of Untrusted Data,15
1,http://example.org/vulnkg/cwe/CWE-787,Out-of-bounds Write,14
2,http://example.org/vulnkg/cwe/CWE-416,Use After Free,12
3,http://example.org/vulnkg/cwe/CWE-78,Improper Neutralization of Special Elements us...,10
4,http://example.org/vulnkg/cwe/CWE-94,Improper Control of Generation of Code ('Code ...,8
5,http://example.org/vulnkg/cwe/CWE-306,Missing Authentication for Critical Function,7
6,http://example.org/vulnkg/cwe/CWE-288,Authentication Bypass Using an Alternate Path ...,6
7,http://example.org/vulnkg/cwe/CWE-284,Improper Access Control,5
8,http://example.org/vulnkg/cwe/CWE-22,Improper Limitation of a Pathname to a Restric...,5
9,http://example.org/vulnkg/cwe/CWE-20,Improper Input Validation,5


In [ ]:
#CWE with longer chains of relationships
longchain= """
PREFIX vuln: <http://example.org/vulnkg/vocab/>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>

SELECT ?cwe (COUNT(?ancestor) AS ?depth)
WHERE {
    ?cwe a vuln:Weakness .
    ?cwe skos:broader+ ?ancestor .
}
GROUP BY ?cwe
ORDER BY DESC(?depth)
LIMIT 3
"""
run_sparql_to_df(g, longchain)

,cwe,depth
0,http://example.org/vulnkg/cwe/CWE-43,13
1,http://example.org/vulnkg/cwe/CWE-45,13
2,http://example.org/vulnkg/cwe/CWE-415,12
